# 11 · Visualisation: lift and cost curves (handbook 3.6)
The tutorial's lift and cost curves need a **yes/no** target and a **score** per building. The EPC label has
8 classes, so the curves are drawn for a business question:

> **Which buildings are energy-poor (E, F or G)?** A renovation programme or a bank wants to find them first.

- **Score per building** = the model's probability of E + F + G (for SVMs: a softmax of the decision values, which ranks buildings but is not a calibrated probability).
- **Lift at p%** = share of E/F/G among the p% highest-scored buildings ÷ share of E/F/G overall.
- **Cost at p%** = cost when the top p% is flagged as energy-poor. The DSC has no official cost matrix, so the
  tutorial's structure is kept as an **assumption to state in the report**: missing an energy-poor building costs 5,
  flagging a good one costs 1.

Change `GROUP` to `GOOD` (A+/A/B) for the green-mortgage version of the question.

In [ ]:
import sys
from pathlib import Path

# find the project root (the folder that contains src/dsc_utils.py) and import the shared helpers
root = Path.cwd()
while not (root / "src" / "dsc_utils.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dsc_utils import *
d = load_prepared()
y_test = d["y_test"]
X_test_hgb = d["data"].iloc[d["indices_test"]]          # HistGradientBoosting uses the cleaned DataFrame

GROUP = POOR                                            # or GOOD
y_binary = np.isin(y_test, GROUP).astype(int)
print(f"Share of {'/'.join(GROUP)} in the test set: {y_binary.mean():.3f}")

## Exercise 1 · Load the saved models and compute their scores on the test set

In [ ]:
scores = {}
for name in MODEL_NAMES:
    if not (OUT / f"model_{name}.pkl").exists():
        print("not trained yet:", name); continue
    model = load_model(name)
    X = X_test_hgb if name == "HGB" else d["x_test"]
    scores[name] = group_score(model, X, GROUP)
    print("scored:", name)

## Exercise 2 · Lift curves for all models, with the random baseline

In [ ]:
P = np.arange(1, 101) / 100
lift_curves = {name: [liftatp(s, y_binary, p) for p in P] for name, s in scores.items()}

plt.figure(figsize=(9, 5))
for name, curve in lift_curves.items():
    plt.plot(P * 100, curve, label=name)
plt.axhline(y=1, color="r", linestyle="--", label="Random baseline")
plt.xlim([0, 100]); plt.ylim([0, max(max(c) for c in lift_curves.values()) + 0.5])
plt.title(f"Lift curve: finding {'/'.join(GROUP)} buildings")
plt.xlabel("Percentage of test buildings targeted (highest score first)"); plt.ylabel("Lift")
plt.legend(loc="upper right"); plt.show()

pd.DataFrame({name: {f"lift at {k}%": round(c[k - 1], 2) for k in [1, 5, 10, 20, 50]}
              for name, c in lift_curves.items()}).T

**Interpretation.** The red line (lift = 1) is what targeting buildings **at random** gives: the share of
E/F/G in the selection equals the overall share. A lift of 2.5 at 10% means the model's top 10% contains 2.5 times
as many energy-poor buildings as a random 10%. Every curve ends at 1 at 100%: targeting everyone is the same as random.
The maximum possible lift is 1 ÷ (share of E/F/G), reached only if the top of the list is entirely E/F/G.

## Exercise 3 · Cost curves for all models

In [ ]:
COST_MATRIX = (0, 1, 5, 0)      # (TN, FP, FN, TP): ASSUMPTION, missing an energy-poor building costs 5x a false alarm

cost_curves = {name: [costatp(s, y_binary, p, COST_MATRIX) for p in P] for name, s in scores.items()}

plt.figure(figsize=(9, 5))
for name, curve in cost_curves.items():
    plt.plot(P * 100, curve, label=name)
plt.title("Cost curve (cost matrix: FP = 1, FN = 5)")
plt.xlabel("Percentage of test buildings targeted (highest score first)"); plt.ylabel("Total cost")
plt.legend(loc="upper right"); plt.show()

pd.DataFrame({name: {"lowest cost": min(c), "at % targeted": int(P[int(np.argmin(c))] * 100)}
              for name, c in cost_curves.items()}).T.sort_values("lowest cost")

**Interpretation.** Targeting nobody (0%) costs 5 × every energy-poor building (all missed); targeting everyone (100%)
costs 1 × every other building (all false alarms). In between, each model's curve has a minimum: the best
percentage to target. The model with the **lowest minimum** is the cheapest under this cost matrix. Because a miss costs
more than a false alarm, the minimum usually lies at a percentage **above** the actual share of E/F/G.